# Lakehouse Agent - Optional Cleanup

This notebook cleans up the resources created by notebooks **01–09**, for the **selected IdP** (`IDP_PROVIDER`).

**⚠️ WARNING: This will delete all resources created during deployment!**

> **On the `[OKTA]` path this is NOT AWS-only — it deletes objects in your Okta tenant too** (Step 8). That is deliberate: cleanup is a full reset on both paths. **Skip Step 8 alone** if you want the AWS side removed and your Okta objects kept.
>
> **Two Okta-specific things to know before you run:** deletion there is **by name**, so a pre-existing object that merely shared a label is deleted with the rest; and your **`OKTA_API_TOKEN` is not removed and cannot be** — you created it, so you have to revoke it. Both are spelled out at Step 8 and repeated in the final summary.

Each deployment step has a dedicated cleanup script. This notebook runs them in **reverse deployment order**, with `## [COGNITO]` / `## [OKTA]` guards on the IdP-specific steps (the flag is read once, below).

**Resources torn down:**
- Identity Provider — **[COGNITO]** User Pool + domain + post-auth Lambda + login-audit table, or **[OKTA]** apps + auth server + groups + users
- **GW1** claims gateway + REQUEST/RESPONSE interceptors + DynamoDB tenant-role map
- **GW2** notes gateway + target + role + OBO/M2M credential providers (+ agent-IAM revert)
- **[COGNITO]** notes REQUEST interceptor (Lambda + role + log group)
- **4a** lakehouse MCP runtime + **4b** OpenSearch MCP runtime (IAM/ECR/CodeBuild each)
- **AOSS** collection `lakehouse-claim-notes` + its encryption/network/data policies
- **S3 Tables** + Lake Formation registration (**pre-existing/shared LF admins are preserved** — fork B17)
- **IAM** tenant roles + LF data-access role
- **S3** bucket (optional) + all **SSM** parameters under `/app/lakehouse-agent/`

**Prerequisites:**
- AWS credentials configured
- Python 3.10 or later
- **[OKTA]** only: `OKTA_ORG_URL` + `OKTA_API_TOKEN` set in `.env` (required by `cleanup_okta.py`)

In [ ]:
# AWS Initialization
from utils.notebook_init import init_aws
from utils.idp_config import get_idp_provider
import subprocess
import sys

session, region, account_id = init_aws()
ssm_client = session.client("ssm", region_name=region)

# Read the IdP flag ONCE; every guard below branches on this variable
# (no per-cell SSM re-read).
IDP_PROVIDER = get_idp_provider(ssm_client)

print("✅ Ready to clean up")
print(f"   Account ID: {account_id}")
print(f"   Region: {region}")
print(f"   IdP Provider: {IDP_PROVIDER}")

## Step 1: Delete Lakehouse Agent Runtime

Deletes the agent runtime, IAM role, ECR repository, and CodeBuild project. Applies to both IdPs (the agent is IdP-agnostic).

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    [sys.executable, "cleanup_agent.py"],
    cwd="deployment/6-lakehouse-agent",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 2: Delete Notes Gateway (GW2), OpenSearch OBO/M2M & Notes Interceptor

`06_cleanup_obo_gateway.py` (**both IdPs**) tears down the GW2 notes gateway + target + IAM role, **both** OAuth2 credential providers (Okta OBO `lakehouse-obo-okta-provider` + Cognito M2M `lakehouse-notes-cognito-oauth-provider`, each safe-if-absent), the **AOSS** collection `lakehouse-claim-notes` + its policies, and reverts the agent IAM OBO patch. The Okta-only bits are `⏭️` no-ops on Cognito and vice-versa.

> ⏳ **The AOSS collection delete blocks until the collection is fully removed (~10 min).** This is expected — let the cell run.

Then, on **## [COGNITO]** only, `interceptor-notes/cleanup.sh` removes the notes REQUEST interceptor (Lambda `lakehouse-notes-interceptor` + role + log group + SSM). Okta has no notes interceptor, so this step is skipped there.

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    [sys.executable, "06_cleanup_obo_gateway.py"],
    cwd="deployment/5b-obo-gateway-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

In [ ]:
## [COGNITO] — Notes REQUEST interceptor teardown (skipped on Okta)
if IDP_PROVIDER == "cognito":
    # No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
    result = subprocess.run(
        ["bash", "cleanup.sh"],
        cwd="deployment/5a-gateway-setup/interceptor-notes",
    )
    if result.returncode != 0:
        print(f"⚠️  Errors above (returncode {result.returncode})")
else:
    print("⏭️  [OKTA] No Cognito notes interceptor to delete — skipping")

## Step 3: Delete Claims Gateway (GW1) & Interceptors

Deletes the GW1 claims gateway + targets, the REQUEST + RESPONSE interceptor Lambdas and their IAM role, the OAuth2 providers, the **DynamoDB tenant-role mapping table**, and the gateway role. Applies to both IdPs (GW1 topology is identical on both paths).

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    [sys.executable, "cleanup_gateway.py"],
    cwd="deployment/5a-gateway-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 4: Delete MCP Server Runtimes (lakehouse 4a + OpenSearch 4b)

Deletes both MCP server runtimes and their IAM roles, ECR repositories, and CodeBuild projects. Both are IdP-agnostic (deployed on both paths).

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    [sys.executable, "cleanup_runtime.py"],
    cwd="deployment/4a-mcp-lakehouse-server",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    [sys.executable, "cleanup_runtime.py"],
    cwd="deployment/4b-mcp-opensearch-server",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 5: Revoke Lake Formation Grants

Revokes the Lake Formation permissions held by the three tenant roles **and** `LakeFormationS3TablesDataAccessRole` — every principal Step 7 deletes.

> ⚠️ **This must run before Steps 6 and 7, and the order is not cosmetic.** Step 6 (`cleanup_s3tables.py`) deletes the `claims`/`users` tables and the SSM keys `table-bucket-name` and `namespace`; those identifiers are required to locate the grants by resource. Step 7 (`cleanup_iam_roles.py`) deletes the grant principals themselves. Run either one first and the grants become orphaned — keyed to a principal ARN that no longer resolves — while the teardown still reports success.

> 📊 The script reports a denominator: **grants found / revoked / refused / remaining after a re-read**, plus a read-error count. A failed query is reported as **UNRESOLVED, never as "no grants"** — see the note in `revoke_lakeformation_permissions.py` on why `list_permissions` must be Resource-qualified.

In [ ]:
# No capture_output: stdout/stderr stream live so the revoke and its re-read show progress.
# Runs BEFORE cleanup_s3tables.py — it needs the SSM identifiers that script deletes.
result = subprocess.run(
    [sys.executable, "revoke_lakeformation_permissions.py"],
    cwd="deployment/3-s3tables-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")
    print("    returncode 1 = grant state UNKNOWN or identifiers unresolved (nothing revoked)")
    print("    returncode 2 = at least one revoke call failed")
    print("    Resolve before running Steps 6 and 7, or grants will be orphaned.")

## Step 6: Delete S3 Tables & Lake Formation

Deletes the S3 Tables bucket, namespace, tables, federated catalog, and **deregisters** the S3 Tables resource from Lake Formation. 

> 🔒 **B17:** this step does **not** remove any Lake Formation *administrators* — pre-existing/shared LF admins are preserved (no `put_data_lake_settings` on teardown; `deregister_resource` only).

> ⚠️ Requires Step 5 to have run first — this step deletes the identifiers the revoke needs.

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    [sys.executable, "cleanup_s3tables.py"],
    cwd="deployment/3-s3tables-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 7: Delete IAM Tenant Roles

Deletes the policyholders, adjusters, and administrators tenant roles plus the Lake Formation data-access role. Applies to both IdPs.

> ⚠️ These four roles are Lake Formation grant principals. **Step 5 must have run first** — deleting a role that still holds grants leaves the grant keyed to a principal ARN that no longer resolves, and this script cannot detect that.

In [ ]:
# No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
result = subprocess.run(
    [sys.executable, "cleanup_iam_roles.py"],
    cwd="deployment/2-lakehouse-tenant-roles-setup",
)
if result.returncode != 0:
    print(f"⚠️  Errors above (returncode {result.returncode})")

## Step 8: Delete Identity Provider

Branches on `IDP_PROVIDER`:
- **## [COGNITO]** — `cleanup_cognito.py`: User Pool + domain, post-auth Lambda + role, login-audit DynamoDB table.
- **## [OKTA]** — `cleanup_okta.py`: OIDC app + OBO exchange app, custom auth server, groups, test users, `okta-*` SSM. **Requires `OKTA_ORG_URL` + `OKTA_API_TOKEN` in `.env`** (the script exits 1 without them), so it is **only** ever run on the Okta path.

> ### 🔑 [OKTA] This step reaches into YOUR Okta tenant — read before running
>
> Cleanup is a **full reset on both paths**, not an AWS-only one. On the Okta path this step deletes the apps `lakehouse-agent-app` and `lakehouse-obo-exchange-client`, the authorization server `lakehouse-agent`, the groups `policyholders` / `adjusters` / `administrators`, the five `@example.com` test users, and the twelve `okta-*` SSM keys.
>
> **Want the AWS side gone but your Okta objects kept? Skip this cell.** Every other step in this notebook is Okta-independent, so nothing downstream breaks.
>
> **⚠️ Deletion is by NAME, not by ownership.** `cleanup_okta.py` matches the same labels `setup_okta.py` adopts, and nothing records which objects the sample actually created. If your deploy log printed `ℹ️  … already exists` — most plausibly for a group called `administrators` — that pre-existing object is deleted here too. See [`deployment/1-okta-setup/README.md`](deployment/1-okta-setup/README.md).
>
> **🔑 Your `OKTA_API_TOKEN` is NOT removed and cannot be.** You minted it by hand in the Okta console, so the sample never owned it. This step deletes the `okta-api-token` **SSM copy**; the credential itself stays live with full admin reach over your tenant. **Revoke it yourself** at Security → API → Tokens, and drop it from your local `.env`. It is listed again in the final summary.

In [ ]:
## [COGNITO] — Cognito teardown (skipped on Okta)
if IDP_PROVIDER == "cognito":
    # No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
    result = subprocess.run(
        [sys.executable, "cleanup_cognito.py"],
        cwd="deployment/1-cognito-setup",
    )
    if result.returncode != 0:
        print(f"⚠️  Errors above (returncode {result.returncode})")
else:
    print("⏭️  [OKTA] Skipping Cognito teardown")

In [ ]:
## [OKTA] — Okta teardown (skipped on Cognito; needs OKTA_ORG_URL + OKTA_API_TOKEN)
if IDP_PROVIDER == "okta":
    # No capture_output: stdout/stderr stream live so the multi-minute delete shows progress.
    result = subprocess.run(
        [sys.executable, "cleanup_okta.py"],
        cwd="deployment/1-okta-setup",
    )
    if result.returncode != 0:
        print(f"⚠️  Errors above (returncode {result.returncode})")
else:
    print("⏭️  [COGNITO] Skipping Okta teardown")

## Step 9: Delete S3 Bucket (Optional)

**⚠️ This permanently deletes all data in the S3 bucket!**

Disabled by default. Set `DELETE_S3_BUCKET = True` to enable.

In [ ]:
DELETE_S3_BUCKET = False  # Change to True to permanently delete the S3 bucket + all its data

if not DELETE_S3_BUCKET:
    print("⏭️  S3 bucket deletion is DISABLED")
    print("   Set DELETE_S3_BUCKET = True to enable")
else:
    s3_client = session.client("s3", region_name=region)
    try:
        bucket_name = ssm_client.get_parameter(Name="/app/lakehouse-agent/s3-bucket-name")["Parameter"]["Value"]
        print(f"Deleting all objects in: {bucket_name}")
        paginator = s3_client.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=bucket_name):
            if "Contents" in page:
                objects = [{"Key": obj["Key"]} for obj in page["Contents"]]
                s3_client.delete_objects(Bucket=bucket_name, Delete={"Objects": objects})
        s3_client.delete_bucket(Bucket=bucket_name)
        print(f"✅ Deleted S3 bucket: {bucket_name}")
    except Exception as e:
        print(f"❌ Error: {e}")

## Step 10: Delete All SSM Parameters

Bulk-deletes every parameter under `/app/lakehouse-agent/` — catches per-user sub keys (`cognito-user-*-sub` / `okta-user-*-sub`), `notes-gateway-*`, `notes-interceptor-lambda-arn`, `idp-provider`, and everything else the scripts above may have left with `--keep-ssm`.

In [ ]:
# Sweep the sample's OWN SSM prefix. FINAL's teardown is a dual-IdP FULL RESET: everything
# under /app/lakehouse-agent/ is deleted, INCLUDING okta-* keys. That is a deliberate
# divergence from the fork's teardown, which preserved okta-* inside its own prefix behind a
# FULL_DECOMMISSION flag. Do not reintroduce that flag here -- different contract.
#
# Every count below carries its denominator, and the delete is VERIFIED by re-reading rather
# than assumed from the call succeeding: 'deleted' and 'still present' must never be
# indistinguishable. delete_parameters() also reports names it REFUSED in InvalidParameters,
# which a bare success message would hide.
OWNED_PREFIX = "/app/lakehouse-agent/"  # trailing slash is LOAD-BEARING (see SIBLING_PREFIX)
SIBLING_PREFIX = "/app/lakehouse-agent-obo/"  # NOT swept: the slash above excludes it


def _names_under(prefix):
    """Enumerate parameter names under a prefix. Raises -- a failed read is never a zero."""
    names = []
    pages = ssm_client.get_paginator("describe_parameters").paginate(
        ParameterFilters=[{"Key": "Name", "Option": "BeginsWith", "Values": [prefix]}]
    )
    for page in pages:
        names.extend(p["Name"] for p in page["Parameters"])
    return names


print(f"🗑️  Sweeping SSM parameters under {OWNED_PREFIX}\n")
try:
    found = _names_under(OWNED_PREFIX)

    deleted, rejected = [], []
    for i in range(0, len(found), 10):
        batch = found[i : i + 10]
        resp = ssm_client.delete_parameters(Names=batch)
        deleted.extend(resp.get("DeletedParameters", []))
        rejected.extend(resp.get("InvalidParameters", []))

    for p in sorted(deleted):
        print(f"   ✅ Deleted:  {p}")
    for p in sorted(rejected):
        print(f"   ⚠️  REFUSED:  {p}  (API rejected the delete)")

    # Verify against AWS, not against our own optimism.
    remaining = _names_under(OWNED_PREFIX)
    preserved = _names_under(SIBLING_PREFIX)

    print()
    print("   " + "-" * 56)
    print(f"   found     : {len(found):>3}  under {OWNED_PREFIX}")
    print(f"   deleted   : {len(deleted):>3}  (confirmed by the API response)")
    print(f"   refused   : {len(rejected):>3}")
    print(f"   remaining : {len(remaining):>3}  (re-read after delete; expected 0)")
    print(f"   preserved : {len(preserved):>3}  under {SIBLING_PREFIX}")
    print("   " + "-" * 56)

    if preserved:
        print(f"\n   🔒 PRESERVED BY DESIGN -- {SIBLING_PREFIX} is NOT part of this sample:")
        for p in sorted(preserved):
            print(f"      • {p}")
        print("      Untouched because the trailing slash in OWNED_PREFIX excludes them.")
        print("      NOTE: this prefix can hold LIVE credentials (e.g. an IdP API token).")
        print("      They still exist after this teardown and are still valid -- delete")
        print("      them separately if you want them gone.")

    if not found:
        print(f"\n⏭️  Nothing to delete: 0 parameters matched {OWNED_PREFIX}.")
        print("   (This is an EMPTY RESULT, not a failed read -- the enumeration ran and")
        print("    returned nothing. A read that fails raises instead, see the handler below.)")
    elif remaining or rejected:
        print(f"\n❌ INCOMPLETE: {len(remaining)} parameter(s) still present under {OWNED_PREFIX}.")
        for p in sorted(remaining):
            print(f"      ! {p}")
        print("   Re-run this cell, or delete the listed names manually. Do NOT treat this")
        print("   teardown as complete.")
    else:
        print(f"\n✅ Deleted {len(deleted)}/{len(found)} parameter(s) under {OWNED_PREFIX};")
        print(f"   0 remaining (verified), {len(preserved)} preserved under {SIBLING_PREFIX}.")
except Exception as e:
    print(f"❌ SSM sweep FAILED: {e}")
    print("   Counts above (if any) are INCOMPLETE. This is a failed read/delete, which is")
    print("   NOT the same as an empty prefix -- do not record this run as a clean teardown.")

## Summary

In [ ]:
print("=" * 60)
print("🎉 CLEANUP COMPLETE")
print("=" * 60)
print()
print(f"IdP Provider: {IDP_PROVIDER}")
print()
print("Shared resources cleaned up (both IdPs):")
print("  • Lakehouse Agent Runtime + IAM role + ECR")
print("  • GW2 notes gateway + target + role + OBO/M2M providers (+ agent-IAM revert)")
print("  • GW1 claims gateway + REQUEST/RESPONSE interceptors + DynamoDB tenant-role map")
print("  • MCP runtimes: 4a lakehouse + 4b OpenSearch (IAM/ECR/CodeBuild each)")
print("  • AOSS collection lakehouse-claim-notes + policies")
print("  • S3 Tables + Lake Formation deregistration (LF admins PRESERVED — B17)")
print("  • IAM tenant roles + LF data-access role")
print("  • S3 bucket (if enabled) + SSM parameters under /app/lakehouse-agent/")
print("    (/app/lakehouse-agent-obo/ is NOT swept -- see the SSM step's summary)")
print()
if IDP_PROVIDER == "cognito":
    print("Cognito-specific resources cleaned up:")
    print("  • Cognito User Pool + domain + post-auth Lambda + login-audit table")
    print("  • Notes REQUEST interceptor (Lambda + role + log group)")
else:
    print("Okta-specific resources cleaned up (IN YOUR OKTA TENANT):")
    print("  • Okta OIDC app + OBO exchange app + auth server + groups + users")
    print("    Deleted BY NAME, not by ownership -- an object that merely shared a")
    print("    label (e.g. a pre-existing 'administrators' group) was deleted too.")
print()
print("Manual cleanup (if needed):")
print("  • CloudWatch Log Groups: /aws/bedrock-agentcore/runtime/*")
print("  • CloudWatch Log Groups: /aws/lambda/lakehouse-*")
if IDP_PROVIDER == "okta":
    # The ONE Okta object this teardown cannot remove. The reader minted the API
    # token by hand in the Okta console, so the sample never owned it and has no
    # way to revoke it -- deleting the okta-api-token SSM entry removes a COPY.
    # Same disclosure shape as the SSM step's PRESERVED BY DESIGN block: never let
    # a deliberate survivor look like a cleanup that quietly failed.
    print()
    print("  🔑 REQUIRED, and only you can do it -- REVOKE YOUR OKTA API TOKEN")
    print("     Okta admin console -> Security -> API -> Tokens -> revoke")
    print("     Then remove OKTA_API_TOKEN from your local .env.")
    print("     Why this is not automated: you created the token by hand, so the")
    print("     sample never owned it. The okta-api-token SSM key deleted above was")
    print("     a COPY -- the credential itself is still live and still carries full")
    print("     administrative reach over your Okta tenant.")